# 03 — Fine-tune ba Transformer và demo C/D
**C1 Quốc Khánh: BERT cased · C2 Đức Trí: RoBERTa · C3 Nhật Huy: DistilBERT + demo.**

Notebook hướng dẫn đọc và kiểm artifact. Các lệnh huấn luyện được in để bạn chạy có chủ ý; mở notebook không tự tải hoặc train 9 models. Chỉ nhận kết quả có artifact thật. [Bài nền tảng GoEmotions](https://aclanthology.org/2020.acl-main.372/).

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Mở notebook từ repo hoặc thư mục notebooks")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.neural import ARCHITECTURES, training_config, run_folder, load_transformer_run
print("Repo:", ROOT)
labels = json.loads((ROOT / "data/labels.json").read_text(encoding="utf-8"))
print("28 nhãn theo đúng thứ tự:", labels)

## 1. Vì sao đây là đa nhãn?
Một câu có thể vừa vui (`joy`) vừa cảm ơn (`gratitude`). Nhãn thật là vector multi-hot 28 cột, ví dụ hai vị trí trên đều bằng 1.

**Văn bản → tokenizer riêng → encoder pretrained → 28 logits → sigmoid → ngưỡng độc lập.**
Huấn luyện dùng **BCEWithLogitsLoss** với nhãn float; không dùng softmax/argmax.
`neutral` là nhãn trong dataset, không tự gán khi model không chọn được nhãn.

Phần A của Duy dùng TF-IDF + Logistic Regression để làm mốc so sánh. C học biểu diễn ngữ cảnh bằng fine-tuning.

In [ ]:
# Cấu hình thật được khai báo trong helper; không chép số vào notebook riêng.
for architecture in ARCHITECTURES:
    print(architecture, training_config(architecture))

## 2. Chạy thử nhỏ trước
Lệnh dưới dùng 64 train, 32 validation, 1 epoch. Chỉ là kiểm luồng. **Không dùng số smoke cho bảng báo cáo cuối.**

Cài PyTorch phù hợp GPU/CPU, rồi requirements-transformer.txt. Xem docs/TRANSFORMERS.md.

In [ ]:
print("python -m scripts.train_transformer --architecture bert --seed 42 --smoke --device auto")
# Chạy dòng in trên ở terminal từ ROOT khi đã sẵn sàng tải model.

## 3. Thực nghiệm chính theo phân công
Mỗi kiến trúc phải có ít nhất ba random seed, cùng cấu hình. Đề xuất: 42, 123, 2026.
Mỗi epoch đánh giá validation @0.5. Giữ checkpoint có Macro-F1 cao nhất; hòa giữ epoch sớm.
Không mở nhãn test để chọn epoch/seed/ngưỡng.

`--resume` bỏ qua run đã hoàn tất, đúng config/hash. Nó không khôi phục optimizer từ giữa epoch.
Checkpoint cased/lr/batch/epoch của C1 tham khảo bài GoEmotions; max tokens/optimizer/warmup và cấu hình C2/C3 là lựa chọn có khai báo của nhóm.

In [ ]:
for architecture in ARCHITECTURES:
    owner = ARCHITECTURES[architecture]["owner"]
    print(owner, "→", f"python -m scripts.run_transformer_seeds --architectures {architecture} --seeds 42 123 2026 --device auto --resume")

## 4. Đọc artifact thật từng seed
Run lưu weights/tokenizer, mapping, config, SHA dữ liệu/checkpoint, history, scores với ID và metrics. Cột dưới chỉ xuất từ run full được kiểm hash. Nếu chưa có run sẽ báo chưa có; không tạo số giả.

In [ ]:
rows = []
for architecture in ARCHITECTURES:
    for seed in (42, 123, 2026):
        folder = run_folder(ROOT, architecture, seed)
        if not (folder / "run_metadata.json").exists():
            print("Chưa có full run:", architecture, seed)
            continue
        run = load_transformer_run(folder, require_full=True)
        rows.append({"architecture": architecture, "seed": seed,
                     "epoch": run["selected_epoch"],
                     "macro_f1_validation": run["metrics"]["macro_f1"],
                     "micro_f1_validation": run["metrics"]["micro_f1"]})
rows

## 5. Mean ± std và best C
Chỉ chọn khi đủ cả ba kiến trúc × ít nhất ba seed.
Mean là mức trung bình, sample std (`ddof=1`) là độ phân tán giữa các seed, không phải khoảng tin cậy.
Chọn kiến trúc theo mean Macro-F1 validation @0.5; hòa ưu tiên std nhỏ hơn rồi ít tham số hơn.
Demo dùng checkpoint seed tốt nhất theo validation của kiến trúc thắng, không phải ensemble.

In [ ]:
print("python -m scripts.select_best_transformer --seeds 42 123 2026")
summary_path = ROOT / "data/processed/transformers/seed_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(json.dumps(summary["summaries"], ensure_ascii=False, indent=2))
else:
    print("Chưa có bảng đủ seed. Chạy huấn luyện + script chọn trước.")

## 6. Phần nâng cao và đánh giá cuối
- `--weighted`: BCE pos_weight = số âm/số dương, chỉ tính bằng train, lưu folder riêng.
- Threshold tuning: chọn ngưỡng trên validation của đúng checkpoint/seed, lưu cùng identity.
- Báo F1 nhãn hiếm trước/sau và false positives; không mặc định nâng cao luôn cải thiện.
- Khóa cấu hình/ngưỡng trước test bằng protocol chung của repo. Bảng cuối phải tách validation/test, smoke/full, standard/weighted và số mỗi seed.
- Đối chiếu ít nhất ba loại lỗi trên cùng ID giữa C1/C2/C3.

Nếu chỉ có một seed ablation weighting, ghi rõ 1 seed; không suy diễn thành mean±std.

## 7. Demo D và minh chứng
App load đúng C full được chọn và kiểm mapping/hash. Không có model thì báo chưa có, không dùng A thay thế hoặc trả kết quả gán sẵn.
Nhập văn bản **tiếng Anh**, hiển thị các nhãn đạt ngưỡng và bảng 28 điểm.
Điểm sigmoid chưa hiệu chuẩn. GoEmotions là Reddit nên không tự xem như đã được kiểm chứng với phản hồi khách hàng tiếng Việt.

In [ ]:
print("python app.py --device auto")
print("Mở http://127.0.0.1:7860 và chụp kết quả tính thật để đưa vào báo cáo.")
print("Đọc hướng dẫn đầy đủ: docs/TRANSFORMERS.md")